# 01 · Why Multiple Agents?

**Module 04 · Multi-Agent Architectures**

So far one assistant did everything: it had a long prompt and several tools. That works for small jobs. As the job grows, one agent gets harder to **control, test and trust**.

The idea of this module: instead of one agent that does everything, build **several small agents, each with one clear job and only the tools it needs**, and make them work together.

In this notebook we do not build anything yet. We look at what "one agent that does everything" costs us.

> 📁 The shop's data and tools (orders, refunds, policy search) come from Modules 02 and 03. They live in `shop.py`, so these notebooks stay short and focus on the new idea: **agents working together**.

In [1]:
import json
from shop import client, MODEL     # the OpenAI client and the model name (see shop.py)

print("Ready")

Ready


## The do-everything agent
It needs all four tools, and a long set of instructions to explain when to use each one.

In [2]:
from shop import LOOKUP_ORDER_TOOL, SEARCH_POLICIES_TOOL, CHECK_REFUND_TOOL, ISSUE_REFUND_TOOL

everything_tools = [LOOKUP_ORDER_TOOL, SEARCH_POLICIES_TOOL, CHECK_REFUND_TOOL, ISSUE_REFUND_TOOL]

everything_instructions = """You are the support agent for an online shop. You handle everything.
- For questions about an order, use lookup_order.
- For questions about shop rules (shipping, warranty, payments), use search_policies and answer only from the results.
- For refund requests: first use lookup_order, then check_refund_eligibility, and only then issue_refund.
- Never issue a refund if the order is not eligible.
- Never promise anything that the tools did not confirm.
- If you do not know something, say so and offer to pass the customer to a human agent.
- Be polite and keep answers to 3 short sentences."""

print("Tools this agent can use:")
for tool in everything_tools:
    print("  -", tool["function"]["name"])

Tools this agent can use:
  - lookup_order
  - search_policies
  - check_refund_eligibility
  - issue_refund


## A specialist agent
Now an agent with **one job**: answering policy questions. It needs one tool and two sentences of instructions.

In [3]:
policy_tools = [SEARCH_POLICIES_TOOL]

policy_instructions = "You answer questions about shop rules using search_policies. Be brief."

print("Tools this agent can use:")
for tool in policy_tools:
    print("  -", tool["function"]["name"])

Tools this agent can use:
  - search_policies


## Same simple question, two agents
Every call carries the agent's **instructions and tool descriptions**. We ask each agent the same question and count the **input tokens** (what we pay to send). We only need the count, so the answer is capped at 50 tokens.

In [4]:
def count_input_tokens(instructions, tools):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": "What is the express shipping fee?"},
        ],
        tools=tools,
        max_tokens=50,
    )
    return response.usage.prompt_tokens

big = count_input_tokens(everything_instructions, everything_tools)
small = count_input_tokens(policy_instructions, policy_tools)

print("Do-everything agent :", big, "input tokens")
print("Policy specialist   :", small, "input tokens")
print("The specialist sends about", round(big / small, 1), "times fewer tokens for the same question.")

Do-everything agent : 337 input tokens
Policy specialist   : 92 input tokens
The specialist sends about 3.7 times fewer tokens for the same question.


Both can answer the question, but one carries a lot more baggage on **every single call**.

## Why split one agent into several?

| Problem with one big agent | What small specialists give you |
|---|---|
| **Long instructions.** Every rule is sent on every call, which costs tokens and can confuse the model. | Short, focused instructions per agent |
| **Many tools.** The more tools to choose from, the easier it is to pick the wrong one. | Each agent sees only 1 to 3 tools |
| **Too much power.** The agent answering a shipping question can also issue refunds. | Only the agents that need `issue_refund` get it |
| **Hard to test.** When something goes wrong, which rule failed? | Test and fix one agent at a time |
| **One model for everything.** | A cheap model for simple jobs, a stronger one only where it matters |

## But more agents is not always better
Every extra agent means **more model calls, more cost, more delay and more places to fail**. If one agent with a few tools does the job well, **keep it**. Split only when the single agent becomes hard to control.

Next: build specialist agents.